In [ ]:
import mercury as mr
app = mr.App(
    title='Quién mueve los hilos de la ciencia mexicana',
    description='Redes de coautoría científica en México 2018-2024',
    show_code=False
)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df_nodos      = pd.read_csv('data/clean/nodos_autores.csv')
df_aristas    = pd.read_csv('data/clean/aristas_colaboraciones.csv')
df_autorias   = pd.read_csv('data/clean/autorias_limpias.csv')
df_pos        = pd.read_csv('data/clean/posiciones.csv')
df_central    = pd.read_csv('data/clean/centralidad.csv')
df_nodos_sub  = pd.read_csv('data/clean/nodos_sub.csv')
df_aristas_sub= pd.read_csv('data/clean/aristas_sub.csv')
df_inst       = pd.read_csv('data/clean/resumen_instituciones.csv')
df_inter      = pd.read_csv('data/clean/colabs_inter_inst.csv')
df_evol_tipo  = pd.read_csv('data/clean/evolucion_tipo.csv')
df_evol_colab = pd.read_csv('data/clean/evolucion_colabs.csv')

BG = '#0d1117'
CARD = '#161b22'
TEXT = '#e6edf3'
GRID = '#21262d'
ACCENT1 = '#8b5cf6'
ACCENT2 = '#3b82f6'
ACCENT3 = '#06b6d4'
ACCENT4 = '#10b981'
ACCENT5 = '#f59e0b'
ACCENT6 = '#ef4444'
ACCENT7 = '#ec4899'
PALETTE = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, ACCENT5, ACCENT6, ACCENT7, '#6366f1', '#14b8a6', '#f97316']

dark_template = go.layout.Template()
dark_template.layout = go.Layout(
    paper_bgcolor=BG,
    plot_bgcolor=CARD,
    font=dict(color=TEXT, family='Inter, system-ui, sans-serif'),
    title=dict(font=dict(size=18, color=TEXT)),
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID),
    colorway=PALETTE,
    margin=dict(l=40, r=40, t=60, b=40),
)

In [ ]:
from IPython.display import HTML
HTML('''
<style>
body, .jp-Notebook, .mercury-app, .mercury-notebook,
#root, .main-content, .container, .container-fluid,
.card, .card-body, .notebook-container,
.jp-Cell, .jp-MarkdownCell, .jp-OutputArea,
div[class*="mercury"], div[class*="notebook"] {
    background-color: #0d1117 !important;
    color: #e6edf3 !important;
}
.navbar, nav, header {
    background-color: #161b22 !important;
    border-bottom: 1px solid #21262d !important;
}
h1, h2, h3, h4, h5, h6, p, li, span, a, label, td, th {
    color: #e6edf3 !important;
}
h1 { color: #c4b5fd !important; }
h2 { color: #a78bfa !important; }
hr { border-color: #21262d !important; }
a { color: #8b5cf6 !important; }
strong, b { color: #c4b5fd !important; }
em, i { color: #9ca3af !important; }
.sidebar, .side-panel, div[class*="sidebar"] {
    background-color: #161b22 !important;
}
</style>
''')

# Quien mueve los hilos de la ciencia mexicana

Cada articulo cientifico es una conversacion. Cuando esas conversaciones se acumulan durante anios, forman **redes invisibles** que definen el poder y alcance de la ciencia de un pais. Este analisis explora la red de coautoria cientifica en Mexico entre **2018 y 2024**, usando datos abiertos de [OpenAlex](https://openalex.org/).

---

### Panorama general

In [ ]:
n_inv = len(df_nodos)
n_colabs = len(df_aristas)
n_inst = df_nodos['inst_principal'].nunique()
n_works = df_autorias['work_id'].nunique()

fig_stats = go.Figure()
vals = [n_inv, n_works, n_colabs, n_inst]
labels = ['Investigadores', 'Articulos', 'Colaboraciones', 'Instituciones']
colors = [ACCENT1, ACCENT2, ACCENT3, ACCENT4]

for i, (v, l, c) in enumerate(zip(vals, labels, colors)):
    fig_stats.add_trace(go.Indicator(
        mode='number',
        value=v,
        title=dict(text=l, font=dict(size=14, color=TEXT)),
        number=dict(font=dict(size=36, color=c)),
        domain=dict(x=[i/4, (i+1)/4], y=[0, 1]),
    ))
fig_stats.update_layout(
    template=dark_template, height=120,
    margin=dict(l=10, r=10, t=10, b=10),
)
fig_stats.show()

---
## Quien domina la produccion cientifica

El siguiente treemap muestra las **15 instituciones con mas investigadores** en la red. El tamanio del bloque representa la cantidad de investigadores; el color indica las **citas acumuladas promedio** — instituciones mas oscuras tienen mayor impacto citado por articulo.

In [ ]:
top15 = df_inst.head(15).copy()
top15['nombre_corto'] = top15['inst_principal'].apply(
    lambda x: x[:35] + '...' if len(str(x)) > 35 else x
)
top15['label'] = top15.apply(
    lambda r: f"{r['nombre_corto']}<br>{r['num_investigadores']} inv. / {r['total_citas']:,.0f} citas", axis=1
)

fig1 = px.treemap(
    top15,
    path=['label'],
    values='num_investigadores',
    color='promedio_citas',
    color_continuous_scale=[[0, '#1e1b4b'], [0.5, ACCENT1], [1, '#c4b5fd']],
    hover_data={'total_articulos': True, 'total_citas': True, 'promedio_citas': ':.0f'},
)
fig1.update_layout(
    template=dark_template, height=500,
    title='Top 15 instituciones por investigadores en la red',
    coloraxis_colorbar=dict(title='Citas prom.', tickfont=dict(color=TEXT)),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig1.update_traces(
    textfont=dict(color='white', size=11),
    marker=dict(cornerradius=5),
)
fig1.show()

---
## Como ha evolucionado la ciencia mexicana

Las areas apiladas muestran el **numero de publicaciones por anio**, separadas por tipo de institucion. La mayoria de la produccion viene de **universidades**, pero los centros gubernamentales y de salud tambien contribuyen significativamente.

In [ ]:
tipo_labels = {
    'education': 'Universidades',
    'facility': 'Centros de inv.',
    'government': 'Gobierno',
    'nonprofit': 'Sin fines de lucro',
    'healthcare': 'Salud',
    'company': 'Empresas',
    'funder': 'Fondos',
    'other': 'Otros',
    'archive': 'Archivos',
}
df_ev = df_evol_tipo.copy()
df_ev['tipo_es'] = df_ev['inst_tipo'].map(tipo_labels).fillna(df_ev['inst_tipo'])

fig2 = px.area(
    df_ev, x='anio', y='publicaciones', color='tipo_es',
    labels={'anio': 'Anio', 'publicaciones': 'Publicaciones', 'tipo_es': 'Tipo'},
    title='Produccion cientifica por tipo de institucion',
    color_discrete_sequence=PALETTE,
)
fig2.update_layout(
    template=dark_template, height=420,
    xaxis=dict(tickmode='linear', dtick=1),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center', font=dict(size=11)),
)
fig2.update_traces(line=dict(width=0.5))
fig2.show()

---
## La red de coautoria

Cada punto es un investigador. Cada linea es una colaboracion publicada. Los colores representan las **instituciones principales** — los puntos mas grandes indican mas conexiones en la red. Pasa el cursor sobre un punto para ver detalles.

In [ ]:
df_plot = df_nodos_sub.merge(df_pos, on='autor_id')
df_plot = df_plot.merge(
    df_central[['autor_id', 'degree', 'betweenness']], on='autor_id', how='left'
)
df_plot['degree'] = df_plot['degree'].fillna(1)

top_inst = df_plot['institucion'].value_counts().head(8).index.tolist()
inst_color_map = {inst: PALETTE[i] for i, inst in enumerate(top_inst)}
df_plot['color'] = df_plot['institucion'].map(inst_color_map).fillna('#444444')
df_plot['inst_short'] = df_plot['institucion'].apply(lambda x: x[:40] + '...' if len(str(x)) > 40 else x)

df_e = df_aristas_sub.merge(
    df_pos.rename(columns={'autor_id':'source','x':'x0','y':'y0'}), on='source')
df_e = df_e.merge(
    df_pos.rename(columns={'autor_id':'target','x':'x1','y':'y1'}), on='target')

edge_x, edge_y = [], []
for _, r in df_e.iterrows():
    edge_x += [r.x0, r.x1, None]
    edge_y += [r.y0, r.y1, None]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.15, color='rgba(139,92,246,0.15)'),
    hoverinfo='none', showlegend=False,
))
fig3.add_trace(go.Scatter(
    x=df_plot['x'], y=df_plot['y'], mode='markers',
    marker=dict(
        size=[max(3, min(14, d**0.7 * 2)) for d in df_plot['degree']],
        color=df_plot['color'],
        opacity=0.9,
        line=dict(width=0.3, color='rgba(255,255,255,0.3)'),
    ),
    text=['<b>' + str(n) + '</b><br>' + str(i) + '<br>' + str(a) + ' arts / ' + str(c) + ' citas'
          for n, i, a, c in zip(df_plot['nombre'], df_plot['inst_short'], df_plot['articulos'], df_plot['citas'])],
    hovertemplate='%{text}<extra></extra>',
    showlegend=False,
))
fig3.update_layout(
    template=dark_template, height=600,
    title=f'Red de coautoria — {len(df_plot)} investigadores mas conectados',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
    plot_bgcolor=BG,
)
fig3.show()

---
## Impacto vs Produccion

Cada burbuja es un investigador del top 80 con mas conexiones. La posicion horizontal indica cuantos **articulos** publico, la vertical cuantas **citas** acumulo, y el tamanio de la burbuja refleja cuantas **conexiones** tiene en la red. El color indica su institucion. Los investigadores arriba a la derecha son los que mas publican y mas impacto generan.

In [ ]:
top80 = df_central.head(80).copy()
top80['inst_short'] = top80['institucion'].apply(lambda x: x[:30] + '...' if len(str(x)) > 30 else x)

fig4 = px.scatter(
    top80,
    x='articulos', y='citas',
    size='degree', color='inst_short',
    hover_name='nombre',
    hover_data={'betweenness': ':.4f', 'degree': True},
    labels={'articulos': 'Articulos publicados', 'citas': 'Citas acumuladas',
            'degree': 'Conexiones', 'inst_short': 'Institucion'},
    title='Impacto vs Produccion — Top 80 investigadores mas conectados',
    color_discrete_sequence=PALETTE,
    size_max=30,
)

# Anotar top 5
for _, row in top80.head(5).iterrows():
    apellido = str(row['nombre']).split()[-1] if pd.notna(row['nombre']) else ''
    fig4.add_annotation(
        x=row['articulos'], y=row['citas'],
        text=apellido, showarrow=True, arrowhead=2,
        font=dict(size=10, color=TEXT),
        arrowcolor='#555',
    )

fig4.update_layout(
    template=dark_template, height=520,
    legend=dict(font=dict(size=9), y=0.5),
)
fig4.show()

---
## Como se conectan las instituciones entre si

Este diagrama muestra los **flujos de colaboracion entre las instituciones mas conectadas**. Las bandas mas gruesas indican mas colaboraciones conjuntas. Revela que alianzas interinstitucionales sostienen la ciencia mexicana.

In [ ]:
top_inst_colabs = (
    df_inter.melt(id_vars=['total_colabs'], value_vars=['inst_source','inst_target'], value_name='inst')
    .groupby('inst')['total_colabs'].sum()
    .sort_values(ascending=False)
    .head(10).index.tolist()
)

df_sankey = df_inter[
    df_inter['inst_source'].isin(top_inst_colabs) &
    df_inter['inst_target'].isin(top_inst_colabs)
].copy()
df_sankey = df_sankey.sort_values('total_colabs', ascending=False).head(25)

short_names = {n: (n[:28] + '...' if len(n) > 28 else n) for n in top_inst_colabs}
node_labels = [short_names[n] for n in top_inst_colabs]
node_map = {n: i for i, n in enumerate(top_inst_colabs)}

src_list, tgt_list, val_list = [], [], []
for _, r in df_sankey.iterrows():
    if r['inst_source'] in node_map and r['inst_target'] in node_map:
        src_list.append(node_map[r['inst_source']])
        tgt_list.append(node_map[r['inst_target']])
        val_list.append(int(r['total_colabs']))

node_colors = PALETTE[:len(node_labels)]

def hex_to_rgba(hex_color, alpha=0.35):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

link_colors = [hex_to_rgba(node_colors[s]) for s in src_list]

fig5 = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=20,
        label=node_labels,
        color=node_colors,
        line=dict(width=0),
    ),
    link=dict(
        source=src_list, target=tgt_list, value=val_list,
        color=link_colors,
    ),
))
fig5.update_layout(
    template=dark_template, height=520,
    title='Flujos de colaboracion inter-institucional',
)
fig5.show()

---
## Que encontramos

La red cientifica mexicana es **densa en el centro, dispersa en los bordes**. Un nucleo de investigadores de la UNAM, CINVESTAV, Tec de Monterrey e IPN concentra la mayoria de conexiones. Los puentes entre instituciones son **pocos pero criticos** — un punado de investigadores sostiene la conexion entre comunidades que de otra forma estarian aisladas.

La pregunta que queda abierta: **como se disenan politicas que incentiven las conexiones correctas?**

---
*Datos: OpenAlex API / CC0 / Analisis: Python, NetworkX, Plotly*